In [1]:
import requests, time, re, csv

# Parameters
BASE_SEARCH = "https://www.loc.gov/search/"
PER_PAGE = 100
MAX_PAGES = 5      # Increase for deeper scan
SLEEP = 0.2
OUTPUT_CSV = "loc_iiif_books.csv"

IIIF_RE = re.compile(r"/iiif/|manifest\.json", re.IGNORECASE)

# Helpers
def iter_strings(obj):
    if isinstance(obj, dict):
        for v in obj.values():
            yield from iter_strings(v)
    elif isinstance(obj, list):
        for v in obj:
            yield from iter_strings(v)
    elif isinstance(obj, str):
        yield obj

def detect_iiif(json_obj):
    """Return (True, sample_url) if anything looks like an IIIF link."""
    for s in iter_strings(json_obj):
        if IIIF_RE.search(s):
            return True, s
    return False, None

def get_json(url, params=None):
    r = requests.get(url, params=params, timeout=20)
    r.raise_for_status()
    return r.json()

# Main loop
total_examined = 0
total_iiif = 0
records = []  # for CSV

print('starting')
for page in range(1, MAX_PAGES + 1):
    params = {
        "fo": "json",
        "c": PER_PAGE,
        "sp": page,
        "fa": "original_format:Book/Printed Material",
    }
    search_json = get_json(BASE_SEARCH, params=params)
    results = search_json.get("results", [])
    print('\nres',len(results))
    if not results:
        break
    
    for res in results:
        item_url = res.get("id")
        title = res.get("title", "")
        if not item_url:
            continue
        item_json_url = item_url + ("&" if "?" in item_url else "?") + "fo=json"
        try:
            item_json = get_json(item_json_url)
        except:
            continue

        total_examined += 1
        has_iiif, sample = detect_iiif(item_json or {})
        if has_iiif:
            total_iiif += 1
            records.append({
                "title": title,
                "loc_url": item_url,
                "iiif_sample": sample or ""
            })

        time.sleep(SLEEP)
    
    print(f"Page {page}: examined={total_examined}, IIIF-ish={total_iiif}")

print('RUNNING')
# Write CSV
with open(OUTPUT_CSV, "w", newline='', encoding="utf-8") as f:
    print('in loop')
    writer = csv.DictWriter(f, fieldnames=["title", "loc_url", "iiif_sample"])
    writer.writeheader()
    writer.writerows(records)

print("\n=== Summary ===")
print(f"Examined items: {total_examined}")
print(f"Likely IIIF-enabled items: {total_iiif}")
print(f"CSV saved to {OUTPUT_CSV}")

starting

res 100
Page 1: examined=91, IIIF-ish=84

res 100
Page 2: examined=183, IIIF-ish=166

res 100
Page 3: examined=274, IIIF-ish=249

res 100
Page 4: examined=362, IIIF-ish=324

res 100
Page 5: examined=445, IIIF-ish=394
RUNNING
in loop

=== Summary ===
Examined items: 445
Likely IIIF-enabled items: 394
CSV saved to loc_iiif_books.csv
